# Lezione 14 — Progetto integrativo

**Corso:** Statistica Applicata con Python · **Blocco 5:** Chiusura
**Tempo stimato:** 120 minuti · **Capitolo del manuale:** `manual/14_proyecto_integrador_IT.md`

## Cosa vedrai in questa lezione

Una **dimostrazione completa** del flusso di un'analisi, dalla domanda al report, con i pinguini di Palmer (non è uno dei tre casi del progetto: così puoi seguire il metodo senza vedere risolto il tuo caso).

1. **Domanda di business → domanda statistica → decisione.**
2. **Audit** dei dati (tipi, mancanti, duplicati, intervalli) ed **esplorazione** prima di modellare.
3. **Scegliere e giustificare il metodo** (ANOVA/Tukey o Kruskal-Wallis; regressione con una dummy) e **verificare i supposti**.
4. **Risultati con incertezza:** effetto + intervallo di confidenza, non solo il p-value.
5. Un **esperimento guidato**: il paradosso di Simpson su dati reali.
6. **Limiti** e un **report di una pagina** scritto in una cella.

## Prerequisiti

- Tutto il corso: lezioni 1-13. Ogni passo rimanda alla lezione in cui lo strumento è stato spiegato; qui **non** si rispiega la teoria.
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già installate in Google Colab).

> **Dati:** il notebook carica `datos/penguins.csv` (Palmer Penguins: Horst, Hill e Gorman, 2020). Se non lo trova, lo scarica con `seaborn`; e se non può nemmeno, genera un dataset **sintetico** con le stesse colonne (i numeri cambieranno). Dopo la lezione, il notebook di pratica usa i tre casi del progetto.

---
## Cella 0 — Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway
from IPython.display import Markdown, display

print("Librerie caricate correttamente.")

---
## 1. La rotta di un'analisi: dalla domanda alla decisione

Un'analisi che comincia con `import pandas` comincia male. Comincia con una **decisione**: qualcuno deve fare (o smettere di fare) qualcosa in base a ciò che scopri. La nostra dimostrazione:

| Passo | Nel nostro caso |
|---|---|
| **Domanda di business** | Una stazione di ricerca vuole riconoscere la specie di un pinguino con misure rapide sul campo. Le specie si distinguono per la massa corporea? |
| **Domanda statistica** | La massa media differisce tra Adelie, Chinstrap e Gentoo? Di quanto, con quale incertezza? |
| **Decisione che informa** | Se la massa separa bene due specie ma non le altre due, la bilancia serve solo per una parte dell'identificazione. |
| **Strumenti** | Audit (lezioni 2-3), ANOVA e post-hoc (lezione 8), bootstrap (lezione 6), regressione (lezione 10), paradosso di Simpson (lezione 13). |

> **Regola della lezione:** prima di ogni cella di codice scrivi cosa ti aspetti di vedere. Alla fine di ogni blocco controlla se avevi ragione.

---
## 2. Audit dei dati

Prima di calcolare qualsiasi cosa, **conosci ciò che hai**: quante righe, quali tipi, cosa manca, cosa si ripete e se gli intervalli hanno senso (lezioni 2 e 3). Definiamo il caricatore (copia locale → `seaborn` → sintetico) ed eseguiamo l'audit.

In [ ]:
def generar_penguins_sintetico(semilla=2026):
    """Dataset SINTETICO con le stesse colonne di `penguins` (esempio illustrativo, NON sono pinguini reali)."""
    r = np.random.default_rng(semilla)
    especies = {
        "Adelie":    dict(n=152, pico=(38.8, 2.7), prof=(18.3, 1.2), aleta=(190, 6.5), masa=(3700, 460), islas=["Torgersen", "Biscoe", "Dream"]),
        "Chinstrap": dict(n=68,  pico=(48.8, 3.3), prof=(18.4, 1.1), aleta=(196, 7.1), masa=(3733, 384), islas=["Dream"]),
        "Gentoo":    dict(n=124, pico=(47.5, 3.1), prof=(15.0, 1.0), aleta=(217, 6.5), masa=(5076, 504), islas=["Biscoe"]),
    }
    bloques = []
    for esp, c in especies.items():
        n = c["n"]
        sexo = r.choice(["Male", "Female"], size=n)
        efecto = np.where(sexo == "Male", 0.5, -0.5)
        def medida(par):
            return par[0] + efecto * par[1] + r.normal(0, par[1] * 0.8, size=n)
        bloques.append(pd.DataFrame({
            "species": esp, "island": r.choice(c["islas"], size=n),
            "bill_length_mm": np.round(medida(c["pico"]), 1), "bill_depth_mm": np.round(medida(c["prof"]), 1),
            "flipper_length_mm": np.round(medida(c["aleta"])), "body_mass_g": np.round(medida(c["masa"]) / 25) * 25,
            "sex": sexo}))
    d = pd.concat(bloques, ignore_index=True)
    idx = r.choice(len(d), size=11, replace=False)
    d.loc[idx[:2], ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g", "sex"]] = np.nan
    d.loc[idx[2:], "sex"] = np.nan
    return d


def cargar_penguins():
    """Restituisce (DataFrame, origine): CSV locale, altrimenti download da seaborn, altrimenti dati sintetici."""
    for ruta in ("../datos/penguins.csv", "datos/penguins.csv", "penguins.csv"):
        if os.path.exists(ruta):
            return pd.read_csv(ruta), "CSV locale"
    try:
        import seaborn as sns
        return sns.load_dataset("penguins"), "seaborn"
    except Exception:
        return generar_penguins_sintetico(), "sintetico"


pen, origen = cargar_penguins()
print("Origine dei dati:", origen)
print("Righe e colonne:", pen.shape)
print()
print(pen.dtypes.to_string())

In [ ]:
auditoria = pd.DataFrame({
    "mancanti": pen.isna().sum(),
    "% mancanti": (pen.isna().mean() * 100).round(1),
    "valori distinti": pen.nunique(),
})
print(auditoria.to_string())
print()
print("Righe duplicate:", int(pen.duplicated().sum()))
print()
print(pen.describe().round(1).T[["count", "min", "50%", "max"]].to_string())
print()
print(pen["species"].value_counts().to_string())

### Cosa osservi?

- Ci sono **344** righe; mancano **2** valori in ogni misura fisica (la stessa coppia di righe) e **11** in `sex` (incluse quelle due). Non ci sono duplicati.
- I mancanti non sono un dettaglio: la regola è **decidere** cosa farne e **dirlo**. Qui non imputeremo; ogni analisi userà le righe complete per le variabili che le servono (`n` = 342 per la massa).
- Gli intervalli sono plausibili (masse da 2 700 a 6 300 g, pinne da 172 a 231 mm): nessun valore impossibile. Se fosse comparsa una massa di 0 o di 60 000, l'analisi sarebbe partita da lì.
- Le specie sono **sbilanciate** (152, 124 e 68): non è un problema per l'ANOVA, ma conviene ricordarlo leggendo gli intervalli.

In [ ]:
print("Righe con almeno un mancante:")
print(pen[pen.isna().any(axis=1)].groupby("species").size().to_string())

def completos(df, columnas):
    """Righe complete per le colonne di un'analisi; dice quante ne scarta."""
    sub = df.dropna(subset=columnas)
    print(f"Analisi con {', '.join(columnas)}: n = {len(sub)} (se ne scartano {len(df) - len(sub)})")
    return sub

masa = completos(pen, ["species", "body_mass_g"])

---
## 3. Esplorare prima di modellare

I grafici e i riassunti per gruppo dicono quale modello ha senso **prima** di adattarlo (lezioni 2 e 3). Qui vogliamo vedere la forma e la dispersione della massa in ogni specie.

In [ ]:
resumen = masa.groupby("species")["body_mass_g"].agg(["count", "mean", "std", "median"]).round(1)
resumen.columns = ["n", "media", "dev. standard", "mediana"]
print(resumen.to_string())

especies = ["Adelie", "Chinstrap", "Gentoo"]
colores = {"Adelie": "tab:orange", "Chinstrap": "tab:purple", "Gentoo": "tab:green"}
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].boxplot([masa.loc[masa["species"] == e, "body_mass_g"] for e in especies], labels=especies)
ejes[0].set_title("Massa corporea per specie"); ejes[0].set_ylabel("grammi")
for e in especies:
    ejes[1].hist(masa.loc[masa["species"] == e, "body_mass_g"], bins=15, alpha=0.55, color=colores[e], label=e)
ejes[1].set_title("Distribuzione della massa"); ejes[1].set_xlabel("grammi"); ejes[1].legend()
plt.tight_layout(); plt.show()

### Cosa osservi?

- **Gentoo** è nettamente sopra (media ≈ 5 076 g); **Adelie** e **Chinstrap** quasi coincidono (≈ 3 701 e 3 733 g): le loro scatole si sovrappongono quasi del tutto.
- La dispersione non è identica (deviazione standard ≈ 459, 384 e 504 g): un avviso per il supposto di varianze uguali.
- Le distribuzioni sono ragionevolmente simmetriche, senza outlier estremi.
- Possiamo già **anticipare** la risposta: Gentoo si separa dal resto; tra Adelie e Chinstrap la massa non aiuta. L'analisi formale deve **quantificarlo con la sua incertezza**, non scoprirlo.

---
## 4. Scegliere e giustificare il metodo

La scelta **non** è un'abitudine: si deduce dalla domanda e dalla natura dei dati. La tabella decisionale che userai nel progetto (dettaglio nel capitolo, sezione 4):

| Domanda | Dati | Metodo | Lezione |
|---|---|---|---|
| Media di un gruppo, con incertezza | Numerica | IC t o bootstrap | 6 |
| Confrontare 2 gruppi numerici | Numerica + 2 gruppi | Welch / Mann-Whitney | 7 |
| Confrontare 3 o più gruppi numerici | Numerica + ≥ 3 gruppi | **ANOVA + Tukey** / Kruskal-Wallis | 8 |
| Associazione tra categorie | Due categoriche | Chi-quadro / Fisher | 9 |
| Molti contrasti insieme | Più p-value | Bonferroni / Holm / BH | 9 |
| Relazione numerica, "a parità del resto" | Risposta numerica | **Regressione lineare** (+ HC3, VIF) | 10-12 |
| Risposta sì/no | Risposta binaria | Regressione logistica | 11 |
| Confrontare una proposta col controllo | Esperimento randomizzato | Test A/B (+ SRM) | 13 |

La nostra domanda (massa di **tre** specie) punta all'**ANOVA**. Prima di fidarci, verifichiamo i suoi supposti (lezione 8): normalità dei residui e omogeneità delle varianze.

In [ ]:
modelo_masa = smf.ols("body_mass_g ~ C(species)", data=masa).fit()
grupos = [masa.loc[masa["species"] == e, "body_mass_g"].values for e in especies]

print("Shapiro-Wilk dei residui: p =", round(stats.shapiro(modelo_masa.resid)[1], 3))
print("Levene (varianze uguali): p =", round(stats.levene(*grupos)[1], 4))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(modelo_masa.resid, line="s", ax=ejes[0]); ejes[0].set_title("QQ dei residui")
ejes[1].scatter(modelo_masa.fittedvalues, modelo_masa.resid, s=10, alpha=0.5)
ejes[1].axhline(0, color="crimson"); ejes[1].set_xlabel("valori adattati"); ejes[1].set_ylabel("residui")
ejes[1].set_title("Residui contro adattati")
plt.tight_layout(); plt.show()

### Cosa osservi?

- **Normalità:** il QQ segue la diagonale e Shapiro dà p ≈ 0,05: al limite, senza deviazioni gravi. Con 342 dati, il teorema del limite centrale (lezione 5) fa il resto.
- **Varianze:** Levene rifiuta l'uguaglianza (p ≈ 0,006). La differenza tra le deviazioni standard (da 384 a 504 g) è moderata, ma il supposto **non è soddisfatto rigorosamente**.
- **Decisione di metodo:** usiamo l'**ANOVA di Welch** (non richiede varianze uguali) e, come controllo, **Kruskal-Wallis** (non richiede normalità). Se coincidono, la conclusione non dipende dal supposto dubbio. Per le coppie, **Tukey** (con la cautela che assume varianze simili) più un **bootstrap** come seconda opinione.

In [ ]:
welch = anova_oneway(grupos, use_var="unequal", welch_correction=True)
kw = stats.kruskal(*grupos)
eta2 = modelo_masa.rsquared                       # proporzione della variazione della massa spiegata dalla specie
print(f"ANOVA Welch: F = {welch.statistic:.1f}, p = {welch.pvalue:.2g}")
print(f"Kruskal-Wallis: H = {kw[0]:.1f}, p = {kw[1]:.2g}")
print(f"Dimensione dell'effetto: eta² = {eta2:.3f}")

tukey = pairwise_tukeyhsd(masa["body_mass_g"], masa["species"], alpha=0.05)
tabla_tukey = pd.DataFrame(tukey.summary().data[1:], columns=tukey.summary().data[0])
print()
print(tabla_tukey.to_string(index=False))

### Cosa osservi?

- Welch e Kruskal-Wallis concordano con forza (p praticamente 0): la massa **differisce** tra le specie. La dimensione dell'effetto è **molto grande**: la specie spiega il 67 % della variazione della massa.
- Ma "differisce" non dice **chi**. Tukey risponde con le differenze a coppie e il loro IC: Gentoo pesa ≈ 1 375 g più di Adelie e ≈ 1 343 g più di Chinstrap, con IC lontani da 0. **Adelie e Chinstrap non si distinguono** (≈ +32 g, IC da −127 a +191 g, che include lo 0).
- Un p-value piccolo senza questo dettaglio avrebbe dato una conclusione fuorviante: "le specie differiscono in massa" non equivale a "le tre sono distinguibili".

In [ ]:
# Seconda opinione: bootstrap percentile della differenza tra medie Gentoo − Adelie (lezione 6)
g_gen = masa.loc[masa["species"] == "Gentoo", "body_mass_g"].values
g_ade = masa.loc[masa["species"] == "Adelie", "body_mass_g"].values
difs = np.array([rng.choice(g_gen, g_gen.size).mean() - rng.choice(g_ade, g_ade.size).mean() for _ in range(5000)])
boot_ic = np.percentile(difs, [2.5, 97.5])
fila = tabla_tukey[(tabla_tukey["group1"] == "Adelie") & (tabla_tukey["group2"] == "Gentoo")].iloc[0]
print(f"Differenza Gentoo − Adelie: {g_gen.mean() - g_ade.mean():.0f} g")
print(f"IC 95 % bootstrap: [{boot_ic[0]:.0f}, {boot_ic[1]:.0f}]   IC 95 % Tukey: [{fila['lower']:.0f}, {fila['upper']:.0f}]")

fig, ax = plt.subplots(figsize=(7, 2.8))
for i, r_ in tabla_tukey.iterrows():
    ax.errorbar(r_["meandiff"], i, xerr=[[r_["meandiff"] - r_["lower"]], [r_["upper"] - r_["meandiff"]]], fmt="o", color="steelblue", capsize=4)
ax.axvline(0, color="crimson", linestyle="--")
ax.set_yticks(range(len(tabla_tukey))); ax.set_yticklabels(tabla_tukey["group1"] + " → " + tabla_tukey["group2"])
ax.set_xlabel("differenza tra medie (g) con IC 95 %"); ax.set_title("Confronti a coppie (Tukey)")
plt.tight_layout(); plt.show()

### Cosa osservi?

- L'IC bootstrap e quello di Tukey sono quasi identici: la conclusione **non dipende** da un metodo specifico. Questo è il valore di una seconda opinione.
- Il grafico riassume la storia meglio di un p-value: due intervalli lontani da 0 (quelli che coinvolgono Gentoo) e uno che lo attraversa (Adelie-Chinstrap).

---
## 5. Regressione con la specie come dummy

Seconda domanda: di quanto è **più lunga** la pinna quando il pinguino pesa di più, **a parità di specie**? È una regressione multipla con una variabile categorica (lezione 10) e errori robusti HC3 per prudenza. Verifichiamo anche la multicollinearità con il VIF (lezione 12).

In [ ]:
aleta = completos(pen, ["species", "body_mass_g", "flipper_length_mm"])
m_simple = smf.ols("flipper_length_mm ~ body_mass_g", data=aleta).fit()
m_mult = smf.ols("flipper_length_mm ~ body_mass_g + C(species)", data=aleta).fit()
m_hc3 = smf.ols("flipper_length_mm ~ body_mass_g + C(species)", data=aleta).fit(cov_type="HC3")

print(f"Senza la specie: +{m_simple.params['body_mass_g'] * 100:.2f} mm di pinna ogni 100 g")
print(f"Con la specie: +{m_hc3.params['body_mass_g'] * 100:.2f} mm ogni 100 g (IC 95 % HC3: {m_hc3.conf_int().loc['body_mass_g', 0] * 100:.2f} a {m_hc3.conf_int().loc['body_mass_g', 1] * 100:.2f})")
print(f"Gentoo rispetto ad Adelie, a parità di massa: +{m_hc3.params['C(species)[T.Gentoo]']:.1f} mm")
print(f"R² corretto: {m_mult.rsquared_adj:.3f}")

X = sm.add_constant(pd.get_dummies(aleta[["body_mass_g", "species"]], drop_first=True).astype(float))
vifs = {col: round(variance_inflation_factor(X.values, i), 2) for i, col in enumerate(X.columns) if col != "const"}
print("VIF:", vifs)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
for e in especies:
    s = aleta[aleta["species"] == e]
    ejes[0].scatter(s["body_mass_g"], s["flipper_length_mm"], s=12, alpha=0.6, color=colores[e], label=e)
ejes[0].set_xlabel("massa (g)"); ejes[0].set_ylabel("pinna (mm)"); ejes[0].legend(); ejes[0].set_title("Pinna e massa per specie")
ejes[1].scatter(m_mult.fittedvalues, m_mult.resid, s=10, alpha=0.5); ejes[1].axhline(0, color="crimson")
ejes[1].set_xlabel("valori adattati"); ejes[1].set_ylabel("residui"); ejes[1].set_title("Diagnostica dei residui")
plt.tight_layout(); plt.show()

### Cosa osservi?

- **Senza controllare la specie** il coefficiente quasi **raddoppia** (+1,53 contro +0,84 mm ogni 100 g): parte della relazione apparente è semplicemente che Gentoo è più pesante **e** ha le pinne più lunghe. È lo stesso meccanismo di confondimento della lezione 10.
- I VIF sono bassi (tra 1,2 e 3,2, tutti sotto 5): massa e specie non sono ridondanti per questo modello.
- I residui non mostrano curvatura né un ventaglio evidente: il modello lineare è ragionevole. L'IC con HC3 è la cifra che comunichiamo.
- Ricorda: questa è un'associazione in dati **osservazionali**. Non diciamo che ingrassare "allunghi" la pinna di un pinguino.

---
## 6. Esperimento guidato: il paradosso di Simpson

La lezione 13 ha mostrato che una tendenza può **invertirsi** mescolando gruppi. Succede con dati reali? Guardiamo la correlazione tra **lunghezza** e **profondità** del becco.

**Prima di eseguire, prevedi:** se disegni tutti i pinguini insieme, la correlazione tra lunghezza e profondità del becco sarà positiva o negativa? E dentro ogni specie?

In [ ]:
pico = completos(pen, ["species", "bill_length_mm", "bill_depth_mm"])
r_total = pico["bill_length_mm"].corr(pico["bill_depth_mm"])
r_esp = {e: pico.loc[pico["species"] == e, "bill_length_mm"].corr(pico.loc[pico["species"] == e, "bill_depth_mm"]) for e in especies}
print(f"Correlazione sul totale: r = {r_total:.3f}")
for e, r_ in r_esp.items():
    print(f"Dentro {e}: r = {r_:.3f}")

fig, ejes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
ejes[0].scatter(pico["bill_length_mm"], pico["bill_depth_mm"], s=12, alpha=0.6, color="gray")
b1, b0 = np.polyfit(pico["bill_length_mm"], pico["bill_depth_mm"], 1)
xs = np.linspace(pico["bill_length_mm"].min(), pico["bill_length_mm"].max(), 50)
ejes[0].plot(xs, b0 + b1 * xs, color="crimson"); ejes[0].set_title(f"Tutti insieme: r = {r_total:.2f}")
for e in especies:
    s = pico[pico["species"] == e]
    ejes[1].scatter(s["bill_length_mm"], s["bill_depth_mm"], s=12, alpha=0.6, color=colores[e], label=e)
    b1, b0 = np.polyfit(s["bill_length_mm"], s["bill_depth_mm"], 1)
    xe = np.linspace(s["bill_length_mm"].min(), s["bill_length_mm"].max(), 20)
    ejes[1].plot(xe, b0 + b1 * xe, color=colores[e])
ejes[1].set_title("Per specie"); ejes[1].legend()
for ax in ejes: ax.set_xlabel("lunghezza del becco (mm)")
ejes[0].set_ylabel("profondità del becco (mm)")
plt.tight_layout(); plt.show()

# Modello multiplo: che segno ha la pendenza includendo la specie?
m_pico = smf.ols("bill_depth_mm ~ bill_length_mm + C(species)", data=pico).fit(cov_type="HC3")
print(f"Pendenza con la specie nel modello: {m_pico.params['bill_length_mm']:+.3f} mm per mm (IC 95 % HC3: {m_pico.conf_int().loc['bill_length_mm', 0]:+.3f} a {m_pico.conf_int().loc['bill_length_mm', 1]:+.3f})")

### Cosa osservi?

- Sul totale la correlazione è **negativa** (r ≈ −0,24): becchi più lunghi sembrano andare con becchi meno profondi. Dentro **ogni specie** è **positiva** (≈ +0,39, +0,65 e +0,64).
- La causa è la composizione: Gentoo ha il becco più lungo **ma** meno profondo; Adelie il più corto e profondo. Mescolando, le differenze **tra** specie dominano e danno la pendenza opposta a quella che esiste **dentro** ciascuna.
- Il modello con la specie recupera la pendenza positiva. **Lezione di progetto:** prima di interpretare un'associazione, chiediti se c'è una variabile di gruppo (segmento, canale, coorte) che mescola popolazioni. Nel caso B del progetto, il paradosso di Simpson e i segmenti sono una trappola reale.

---
## 7. Analisi di sensibilità e limiti

Un risultato è più credibile se **sopravvive** a decisioni ragionevoli diverse. Verifichiamo che la differenza Gentoo − Adelie non dipenda dal mescolare i sessi e dai mancanti di `sex`.

In [ ]:
sens = []
for etiqueta, sub_ in [("Tutti (n con massa)", masa),
                       ("Solo maschi", masa[masa["sex"] == "Male"]),
                       ("Solo femmine", masa[masa["sex"] == "Female"])]:
    a = sub_.loc[sub_["species"] == "Adelie", "body_mass_g"]
    g = sub_.loc[sub_["species"] == "Gentoo", "body_mass_g"]
    t = stats.ttest_ind(g, a, equal_var=False)
    ic = t.confidence_interval(0.95)                  # IC di Welch per la differenza tra medie
    sens.append({"sottoinsieme": etiqueta, "n": len(sub_), "dif. Gentoo−Adelie (g)": round(g.mean() - a.mean()),
                 "IC 95 % inf.": round(ic.low), "IC 95 % sup.": round(ic.high)})
print(pd.DataFrame(sens).to_string(index=False))

### Cosa osservi?

- La differenza è grande (da ≈ 1 310 a ≈ 1 440 g) nei tre sottoinsiemi e gli intervalli non si avvicinano mai a 0: **la conclusione è robusta** a questa decisione.
- Comunque, **limiti** che il report deve dichiarare: (1) i dati sono **osservazionali** e di **tre isole** di un arcipelago e tre anni, non un campione casuale di tutti i pinguini; (2) il supposto di varianze uguali è soddisfatto solo in modo approssimato; (3) ci sono **11** righe con almeno un mancante che non partecipano a ogni analisi; (4) un'associazione tra massa e pinna non è una relazione causale; (5) i pinguini di una stessa colonia non sono strettamente indipendenti.

---
## 8. Il report di una pagina

Un report **non ripete il notebook**: risponde alla domanda, con cifre, incertezza e limiti, in una pagina. Modello: **Domanda e decisione · Dati · Metodo · Risultati · Limiti · Raccomandazione**. Lo scriviamo in una cella che prende le cifre **dalle variabili**, così non ci sono discrepanze tra il testo e l'analisi.

In [ ]:
def num(x, d=0):
    return f"{x:.{d}f}".replace(".", ",")

def fila_tukey(g1, g2):
    return tabla_tukey[(tabla_tukey["group1"] == g1) & (tabla_tukey["group2"] == g2)].iloc[0]

ag, cg, ac = fila_tukey("Adelie", "Gentoo"), fila_tukey("Chinstrap", "Gentoo"), fila_tukey("Adelie", "Chinstrap")
n_masa = len(masa)

informe = f"""
### Report: la massa corporea distingue le tre specie di pinguini?

**Domanda e decisione.** Una misura rapida di massa serve a identificare la specie? Decisione: se includere la bilancia nel protocollo di campo.

**Dati.** {len(pen)} pinguini di Palmer (tre specie); {n_masa} con massa registrata (se ne scartano {len(pen) - n_masa} con mancanti). Nessun duplicato né valore impossibile. Dati osservazionali.

**Metodo.** ANOVA di Welch (varianze diverse secondo Levene, p = {num(stats.levene(*grupos)[1], 3)}), confermata con Kruskal-Wallis; confronti a coppie con Tukey e bootstrap come seconda opinione.

**Risultati.**
- La massa differisce tra le specie (Welch F = {welch.statistic:.0f}, p < 0,001; eta² = {num(eta2, 2)}: la specie spiega il {eta2 * 100:.0f} % della variazione).
- Gentoo pesa {num(ag["meandiff"])} g in più di Adelie (IC 95 %: {num(ag["lower"])} a {num(ag["upper"])}) e {num(cg["meandiff"])} g in più di Chinstrap (IC 95 %: {num(cg["lower"])} a {num(cg["upper"])}).
- Adelie e Chinstrap **non** si distinguono per la massa ({num(ac["meandiff"])} g; IC 95 %: {num(ac["lower"])} a {num(ac["upper"])}, che include lo 0).
- La differenza Gentoo − Adelie si mantiene in maschi e femmine separatamente.

**Limiti.** Dati osservazionali di tre isole, non un campione casuale di tutti i pinguini; varianze solo approssimativamente uguali; osservazioni di una stessa colonia non strettamente indipendenti. I risultati descrivono un'associazione, non una causa.

**Raccomandazione.** Usare la bilancia per riconoscere **Gentoo**, ma non per separare Adelie da Chinstrap: per quella coppia serve un'altra misura (ad esempio la lunghezza del becco).
"""
display(Markdown(informe))

### Cosa osservi?

- Ogni affermazione porta **effetto + IC**, non solo un p-value; i **limiti** sono scritti prima della raccomandazione; e la raccomandazione risponde alla **decisione** posta all'inizio.
- Il report è breve perché decide cosa **non** raccontare: i grafici dei supposti, il bootstrap e le diagnostiche restano nel notebook.

---
## 9. La rubrica di dieci criteri

Quando finisci il tuo progetto, dai un punteggio a ogni criterio da 0 (non fatto) a 2 (fatto bene). Nella dimostrazione:

| # | Criterio | Dove si vede qui? |
|---|---|---|
| 1 | Domanda e decisione definite | Sezione 1 |
| 2 | Audit dei dati | Sezione 2 |
| 3 | Esplorazione prima di modellare | Sezione 3 |
| 4 | Metodo giustificato | Sezione 4 (tabella decisionale) |
| 5 | Supposti verificati | Shapiro, Levene, QQ, residui |
| 6 | Effetto + IC, non solo p-value | Tukey, bootstrap, HC3 |
| 7 | Molteplicità o selezione controllate | Tukey (3 coppie insieme) |
| 8 | Limiti e distorsioni dichiarati | Sezione 7 |
| 9 | Nessuna affermazione causale indebita | "associazione, non causa" |
| 10 | Comunicazione chiara e riproducibile | Seme fisso, report di una pagina |

## Riepilogo

- Un'analisi va dalla **domanda** alla **decisione**; il codice è il mezzo.
- **Fai l'audit** prima di calcolare, **esplora** prima di modellare, **giustifica** il metodo e **verifica** i suoi supposti.
- Comunica **effetto + incertezza + limiti**. Un p-value da solo non è un risultato.
- Diffida delle medie globali quando ci sono gruppi nascosti (Simpson) e delle associazioni osservazionali come cause.

## Adesso tocca a te

Apri `notebooks_practica/clase14_practica_proyecto_integrador_IT.ipynb`, scegli **uno** dei tre casi (A: salute, B: e-commerce con test A/B, C: immobiliare) e portalo fino a un report di una pagina. Con questo **il corso termina**: per proseguire, il passo naturale è un corso di Machine Learning, che riprende la validazione (lezione 12) e i modelli della lezione 11 con strumenti di previsione.